# ReFit — QLoRA-DPO disclaimer calibration

Trains a LoRA adapter on Llama-3.1-8B-Instruct so calibrated disclaimer behaviour becomes the
default **without** the policy prompt (context distillation). Runs on a free Colab **T4**.

**Before you start:**
1. Runtime → Change runtime type → **T4 GPU**.
2. Accept the Llama 3.1 licence on HF (huggingface.co/meta-llama/Llama-3.1-8B-Instruct) — gated.
3. Upload `dpo_pairs.jsonl` (left panel → Files) **or** mount Drive in the cell below.

The bar for Wednesday is the **last cell**: same prompt, base vs base+adapter, behaviour visibly changes.
GGUF/Ollama conversion is a separate, optional step — not needed to prove the method works.

In [ ]:
# 1. Install. Pinned-ish to avoid a surprise TRL API break mid-deadline.
!pip -q install "transformers>=4.44" "trl>=0.11" "peft>=0.12" "datasets>=2.20" \
    "bitsandbytes>=0.43" "accelerate>=0.33"

In [ ]:
# 2. Auth + config. Paste your HF token when prompted (needs the gated Llama licence accepted).
from huggingface_hub import login
login()  # or: login(token='hf_...')

BASE_MODEL = 'meta-llama/Llama-3.1-8B-Instruct'   # full-precision twin of your ollama llama3.1:latest
DATA_PATH  = 'dpo_pairs.jsonl'                     # uploaded file, or a Drive path
OUT_DIR    = 'refit-dpo'                           # adapter output

# Optional: checkpoint to Drive so a Colab disconnect doesn't lose the run.
# from google.colab import drive; drive.mount('/content/drive')
# OUT_DIR = '/content/drive/MyDrive/refit-dpo'

In [ ]:
# 3. Load data. Our JSONL already has prompt / chosen / rejected — the exact DPO schema.
# We template `prompt` as a user turn so the model sees the same chat format the app uses.
# NOTE: no policy prompt goes in — that's the point. The tune must hold the policy without it.
from datasets import load_dataset
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
tok.pad_token = tok.eos_token

def to_prompt(ex):
    ex['prompt'] = tok.apply_chat_template(
        [{'role': 'user', 'content': ex['prompt']}],
        tokenize=False, add_generation_prompt=True)
    return ex

ds = load_dataset('json', data_files=DATA_PATH)['train']
ds = ds.map(to_prompt)
ds = ds.train_test_split(test_size=0.1, seed=42)  # ~50 held out to watch overfit
print(ds)
print('example prompt:\n', ds['train'][0]['prompt'][:300])

In [ ]:
# 4. Load base in 4-bit (QLoRA). fp16 compute — T4 is pre-Ampere, no bf16.
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_use_double_quant=True)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb, device_map='auto', torch_dtype=torch.float16)
model.config.use_cache = False

peft_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'])

In [ ]:
# 5. Train. 500 pairs, 1 epoch ≈ ~56 steps — fast. Watch eval loss for overfit.
from trl import DPOConfig, DPOTrainer

cfg = DPOConfig(
    output_dir=OUT_DIR,
    beta=0.1,                          # preference strength — main dial if results look off
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,     # effective batch 8; T4 can't hold 8B x8
    max_length=1024, max_prompt_length=512,
    num_train_epochs=1,                # DPO overfits fast on small sets; start at 1
    learning_rate=5e-5, lr_scheduler_type='cosine', warmup_ratio=0.1,
    fp16=True, gradient_checkpointing=True,
    logging_steps=5, eval_strategy='steps', eval_steps=10,
    save_strategy='epoch', report_to='none')

trainer = DPOTrainer(
    model, ref_model=None,             # LoRA: reference = adapter disabled, no 2nd model in VRAM
    args=cfg, train_dataset=ds['train'], eval_dataset=ds['test'],
    processing_class=tok, peft_config=peft_cfg)

trainer.train()
trainer.save_model(OUT_DIR)
print('adapter saved to', OUT_DIR)

## Proof cell — this is the Wednesday deliverable

Same held-out prompts through **base** vs **base+adapter**, no policy prompt on either side.
If the adapter row shows calibrated disclaimer behaviour (Tier 0/1 hedging removed, Tier 2/3
deferral kept) where base does not, the method is proven. Screenshot this for Results.

In [ ]:
# 6. Before/after. Pull a few held-out prompts spanning tiers.
from peft import PeftModel

def generate(m, user_prompt):
    msgs = [{'role': 'user', 'content': user_prompt}]
    ids = tok.apply_chat_template(msgs, return_tensors='pt', add_generation_prompt=True).to(m.device)
    out = m.generate(ids, max_new_tokens=300, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids.shape[-1]:], skip_special_tokens=True).strip()

# Raw user text (undo the template) for a handful of held-out cases across tiers.
import json
raw = load_dataset('json', data_files=DATA_PATH)['train']
by_tier = {}
for r in raw:
    by_tier.setdefault(r['tier'], r['prompt'])
probes = [by_tier[t] for t in sorted(by_tier)]

tuned = PeftModel.from_pretrained(model, OUT_DIR)
for p in probes:
    print('=' * 80); print('PROMPT:', p)
    tuned.disable_adapter_layers();  print('\n--- BASE ---\n',  generate(model, p))
    tuned.enable_adapter_layers();   print('\n--- TUNED ---\n', generate(tuned, p))

## Fuse → Ollama (run after the proof cell)

Primary path = **adapter GGUF + Ollama `ADAPTER`**. It skips the fp16 merge, which is ~16GB and does NOT fit on a T4. Ollama loads your existing llama3.1 base plus this small (~100MB) adapter, so the download is tiny. Full-merge fallback is at the bottom if this route misbehaves.

In [ ]:
# 7. Build llama.cpp (for the adapter->GGUF converter).
!git clone --depth 1 https://github.com/ggerganov/llama.cpp
!pip -q install -r llama.cpp/requirements.txt

In [ ]:
# 8. Convert JUST the LoRA adapter to GGUF — small file, no 16GB merge.
# OUT_DIR holds adapter_config.json + adapter_model.safetensors from trainer.save_model.
# NOTE: flag names can drift between llama.cpp versions; if it errors, run
#   !python llama.cpp/convert_lora_to_gguf.py -h   to check --base / --outfile / --outtype.
!python llama.cpp/convert_lora_to_gguf.py {OUT_DIR} --base {BASE_MODEL} --outfile refit-lora.gguf
!ls -lh refit-lora.gguf

In [ ]:
# 9. Download the adapter to your Mac.
from google.colab import files
files.download('refit-lora.gguf')

## On your Mac — create the Ollama model + swap it in

```bash
# 1. Get the base's EXACT template + stop tokens (never hand-write them):
ollama show --modelfile llama3.1 > Modelfile

# 2. Edit Modelfile: change the top `FROM ...` line to `FROM llama3.1` and add the
#    adapter right under it, keeping every TEMPLATE and PARAMETER stop line as-is:
#        FROM llama3.1
#        ADAPTER ./refit-lora.gguf

# 3. Build and smoke-test:
ollama create refit-dpo -f Modelfile
ollama run refit-dpo "can I do squats 3 times a week?"          # Tier 0: should NOT hedge
ollama run refit-dpo "I get chest pain when I run, is that ok?" # Tier 3: should defer to care
```

Then in `pipelines.py`, change ONLY the answerer/rewriter `chat('llama3.1', ...)` calls (lines ~88 / 115 / 264) to `chat('refit-dpo', ...)`. Leave every `structured_chat('llama3.1', ...)` on base — the tune must never touch the JSON stages.

---
**Fallback — full merge** (only if the ADAPTER route misbehaves). Merging needs the base in fp16 (~16GB) → OOMs a T4, so use an A100/high-RAM runtime:
```python
from peft import PeftModel; from transformers import AutoModelForCausalLM
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype='float16')
PeftModel.from_pretrained(base, OUT_DIR).merge_and_unload().save_pretrained('merged'); tok.save_pretrained('merged')
# !python llama.cpp/convert_hf_to_gguf.py merged/ --outfile refit-f16.gguf --outtype f16
# !./llama.cpp/llama-quantize refit-f16.gguf refit-q4_k_m.gguf Q4_K_M   → Modelfile: FROM ./refit-q4_k_m.gguf
```